## upload.py


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from  langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import  Chroma ,FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

c:\Users\AYMAN KHAN\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [105]:
user = "Attention.pdf"

In [109]:
loader = PyPDFLoader(user)

document = loader.load()

print(document[0])


page_content='Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolution

In [64]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_overlap=50,
    chunk_size=500
)

In [110]:
chunks = text_splitter.split_documents(document)

In [111]:
len(chunks)

93

In [108]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [112]:
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

In [113]:
retriever = vector_store.as_retriever(
    
    search_kwargs={"k": 3}
)

## image extraction.py 

In [11]:
import fitz  # PyMuPDF
import os
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from sentence_transformers import SentenceTransformer
from  langchain_community.vectorstores import FAISS
from chromadb.config import Settings


In [68]:
os.makedirs("extracted_images", exist_ok=True)

In [114]:
pdf_path = "Attention.pdf"   

doc = fitz.open(pdf_path)


In [115]:
image_paths = []

for page_index in range(len(doc)):

    page = doc[page_index]

    image_list = page.get_images(full=True)

    for img_index, img in enumerate(image_list):

        xref = img[0]

        base_image = doc.extract_image(xref)

        image_bytes = base_image["image"]

        image_ext = base_image["ext"]

        image_filename = f"extracted_images/page_{page_index+1}_img_{img_index+1}.{image_ext}"

        with open(image_filename, "wb") as f:
            f.write(image_bytes)

        image_paths.append(image_filename)

print(f"Total Images Extracted: {len(image_paths)}")

Total Images Extracted: 3


In [72]:
## loading BLIP model for embeddings 

print("\nLoading BLIP model...")

processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

print("BLIP Loaded Successfully!")


Loading BLIP model...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

BLIP Loaded Successfully!


In [116]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [117]:
## creating faiss index 
import faiss
dimension = 384

index = faiss.IndexFlatL2(dimension)

In [118]:
image_metadata = []

In [119]:

for idx, image_path in enumerate(image_paths):

    print(f"\nProcessing Image {idx+1}")

    # LOAD IMAGE
    raw_image = Image.open(image_path).convert("RGB")

    # IMAGE → CAPTION
    inputs = processor(raw_image, return_tensors="pt")

    output = model.generate(**inputs)

    caption = processor.decode(
        output[0],
        skip_special_tokens=True
    )

    print("Caption:", caption)


Processing Image 1
Caption: a block diagram with the block and block

Processing Image 2
Caption: a diagram of the same algorithm

Processing Image 3
Caption: a diagram of the same product


In [120]:
import numpy as np

embedding = embedding_model.encode(caption)

embedding = np.array([embedding]).astype("float32")

In [121]:
index.add(embedding)

    # STORE METADATA
image_metadata.append({
        "image_path": image_path,
        "caption": caption
    })


In [78]:
query = input("\nEnter Query: ")

# QUERY EMBEDDING
query_embedding = embedding_model.encode(query)

query_embedding = np.array([query_embedding]).astype("float32")

In [79]:
k = 2

distances, indices = index.search(query_embedding, k)

In [122]:
print("\n==========================")
print("TOP RETRIEVED IMAGES")
print("==========================")


for i, idx in enumerate(indices[0]):

    result = image_metadata[idx]

    print(f"\nResult {i+1}")

    print("Image Path:")
    print(result["image_path"])

    print("\nCaption:")
    print(result["caption"])

    print("\nDistance Score:")
    print(distances[0][i])


TOP RETRIEVED IMAGES

Result 1
Image Path:
extracted_images/page_4_img_2.png

Caption:
a diagram of the same product

Distance Score:
1.7754271

Result 2
Image Path:
extracted_images/page_4_img_2.png

Caption:
a diagram of the same product

Distance Score:
3.4028235e+38


In [25]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")

In [140]:
query = "Attention"

In [141]:
retrieved_docs = retriever.invoke(query)

In [142]:
retrieved_docs

[Document(id='77902f2c-e779-4c42-a83d-ac04c71cb837', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Attention.pdf', 'total_pages': 15, 'page': 2, 'page_label': '3'}, page_content='masking, combined with fact that the output embeddings are offset by one position, ensures that the\npredictions for position i can depend only on the known outputs at positions less than i.\n3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3'),
 Document(id='b8781f83-8dc0-44a6-893a-02a46e0d767f', metadata={'producer': 'pdfTeX-1.40.25', '

In [143]:
context =  "\n\n".join(docs.page_content for docs in retrieved_docs )

In [144]:
final_prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}
"""


In [145]:
response = llm.invoke(final_prompt)

In [146]:
print(response.content)

An attention function is a mapping that takes a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum. In other words, attention is a mechanism that allows the model to focus on specific parts of the input data when generating output, by weighing the importance of different input elements.


## multi model rag 

In [147]:
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration
)

from PIL import Image

processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

captions = []

for image_path in image_paths:

    raw_image = Image.open(image_path).convert("RGB")

    inputs = processor(
        raw_image,
        return_tensors="pt"
    )

    output = model.generate(**inputs)

    caption = processor.decode(
        output[0],
        skip_special_tokens=True
    )

    captions.append(caption)

    print(caption)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

a block diagram with the block and block
a diagram of the same algorithm
a diagram of the same product


In [149]:
## storng the image captions in faiss


from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

image_vectorstore = FAISS.from_texts(
    texts=captions,
    embedding=embedding_model
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [150]:
query = "show transformer architecture"

results = image_vectorstore.similarity_search(
    query,
    k=2
)

for doc in results:
    print(doc.page_content)

a diagram of the same product
a diagram of the same algorithm


In [152]:
text_results = vector_store.similarity_search(
    query,
    k=2
)

image_results = image_vectorstore.similarity_search(
    query,
    k=2
)

In [153]:
text_context = "\n".join(
    [doc.page_content for doc in text_results]
)

image_context = "\n".join(
    [doc.page_content for doc in image_results]
)

final_context = f"""
TEXT CONTEXT:
{text_context}

IMAGE CONTEXT:
{image_context}
"""

In [154]:
prompt = f"""
Answer the question using the provided context.

{final_context}

Question:
{query}
"""

In [155]:
reps = llm.invoke(prompt)

In [ ]:
print(reps)

content='The Transformer architecture is shown in Figure 1, which consists of an encoder and a decoder. \n\nThe encoder is composed of a stack of 6 identical layers, each with two sub-layers: \n1. A multi-head self-attention mechanism\n2. A simple, position-wise, fully connected layer\n\nThe decoder also follows a similar architecture with stacked self-attention and point-wise, fully connected layers.\n\nPlease refer to Figure 1 for a visual representation of the Transformer model architecture.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 258, 'total_tokens': 358, 'completion_time': 0.414927031, 'completion_tokens_details': None, 'prompt_time': 0.013186262, 'prompt_tokens_details': None, 'queue_time': 0.056470508, 'total_time': 0.428113293}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019